<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

import sys

#!pip install import-ipynb
#import import_ipynb

#!pip install open3d plotly
#import open3d as o3d

import matplotlib.pyplot as plt


Mounted at /content/drive


In [60]:
class ResNetBlock(nn.Module):

  def __init__(self, in_dims, out_dims):

    super().__init__()
    self.stride_layer = nn.Conv2d(in_dims, out_dims, kernel_size=3, stride = 2, padding =1)
    self.layer1 = nn.Sequential(nn.Conv2d(out_dims, out_dims, kernel_size = 3, padding =1), nn.BatchNorm2d(out_dims), nn.ReLU())
    self.layer2 = nn.Sequential(nn.Conv2d(out_dims, out_dims, kernel_size = 3, padding =1), nn.BatchNorm2d(out_dims))
    self.relu = nn.ReLU()

  def forward(self, x):

    out = self.stride_layer(x)
    delta = self.layer1(out)
    delta = self.layer2(delta)
    out = delta + out
    out = self.relu(out)

    return out

class ResNet(nn.Module):

  def __init__(self, in_dims, in_channels, out_channels, block_channels = [32, 64, 128, 256]):

    super().__init__()

    self.in_channels = in_channels
    self.out_channels = out_channels
    self.block_channels = block_channels

    self.stemlayer = nn.Sequential(nn.Conv2d(in_channels, block_channels[0], kernel_size=7, stride = 2, padding =3), nn.MaxPool2d(3, stride = 2, padding = 1))
    self.block1 = ResNetBlock(block_channels[0], block_channels[1])
    self.block2 = ResNetBlock(block_channels[1], block_channels[2])
    self.block3 = ResNetBlock(block_channels[2], block_channels[3])
    self.pool = nn.AdaptiveAvgPool2d(1)
    self.out = nn.Linear(block_channels[3], out_channels)

  def forward(self, x):

    out = self.stemlayer(x)
    out = self.block1(out)
    out = self.block2(out)
    out = self.block3(out)
    out = torch.flatten(self.pool(out), 1)
    out = self.out(out)

    return out

In [61]:
model = ResNet(512, 3, 64)

In [62]:
rng = np.random.default_rng()

test_mat = rng.random((2,3,512,512))

test_mat = torch.from_numpy(test_mat).to(torch.float)

In [63]:
model(test_mat).shape

torch.Size([2, 64])